# Transformer Foundations, Part 4: A Modern Decoder-Only LLM

> **The story:** Between 2019 and 2023, practical ideas changed the anatomy of dense decoder-only models without changing their job. RMSNorm narrowed normalization to scale, SwiGLU added an independent learned gate, RoPE moved position into attention, and grouped-query attention reduced key/value ownership.
>
> **Where you are:** Parts 1-3 built attention, a GPT-2-style causal language model, and the encoder/decoder architecture map. Riverside can predict the next token in `Aria heard the signal aboard Meridian`, but its teaching block still has older anatomy.
>
> **What this notebook delivers:** A fresh-kernel, CPU-safe `TinyModernLM` plus `artifacts/base-lm/model-config.json`. The objective stays unchanged; only block internals move forward. This chapter uses shapes, plots, animations, and measured checks instead of derivations or a notation inventory.

## 0. Scope and the Architecture Gap

| Sub-topic | Coverage | Why |
|---|---|---|
| RMSNorm, SwiGLU, GQA, RoPE | Built | Missing mechanisms in the modern dense block |
| `ModernDecoderBlock`, `TinyModernLM`, tied embeddings | Built | Notebook 06 needs a compact model contract |
| MHA and MQA | Explained and measured | Comparison endpoints for GQA sharing |
| Representative Llama-style config | Explained | No weights or network access required |
| Production training and inference optimizations | Named only | Different engineering goals, outside this build |

> **The mission:** Riverside House - keep causal next-token prediction, but replace four older block choices with inspectable modern counterparts.

**What we know so far:** the teaching decoder already maps token IDs to causal vocabulary logits. **But** with 8 query heads it stores 8 key heads and 8 value heads, and its block anatomy no longer resembles a current dense Llama-style model.

**Predict:** Can normalization, feed-forward gating, key/value ownership, and position handling all change while `(batch, tokens, vocabulary)` remains fixed?

```mermaid
flowchart LR
    A["Token IDs"] --> B["Tied embedding"] --> C["RMSNorm"] --> D["GQA with RoPE"] --> E["Residual"] --> F["RMSNorm"] --> G["SwiGLU"] --> H["Residual"] --> I["Tied vocabulary head"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Roadmap:** expose the older choice, visualize its replacement, measure the changed contract, implement it, then test what still breaks.

In [ ]:
# -- Imports and deterministic chapter state ----------------------------------
import json
import math
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

SEED = 17
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
TOKENS = ["Aria", "heard", "the", "signal", "aboard", "Meridian"]
DARK, BLUE, AMBER, GREEN, RED = "#1a1a2e", "#60a5fa", "#f59e0b", "#22c55e", "#ef4444"
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": DARK, "axes.facecolor": DARK})
print(f"torch={torch.__version__}; numpy={np.__version__}; seed={SEED}")
print("  -> Fresh-kernel state is ready; every quoted comparison is deterministic.")

In [ ]:
# -- Compare block inventories while holding the objective fixed --------------
inventory = [
    ("Teaching MiniLM", "LayerNorm", "MHA", "learned positions", "GELU FFN"),
    ("GPT-2 style", "LayerNorm", "MHA", "learned positions", "GELU FFN"),
    ("Small Llama style", "RMSNorm", "GQA", "RoPE in attention", "SwiGLU"),
]
headers = ("Model", "Normalization", "Attention", "Position", "FFN")
widths = [22, 16, 12, 20, 12]
print("".join(text.ljust(width) for text, width in zip(headers, widths)))
print("-" * sum(widths))
for row in inventory:
    print("".join(text.ljust(width) for text, width in zip(row, widths)))
input_contract = (2, len(TOKENS))
output_contract = (2, len(TOKENS), 64)
print(f"\nObjective contract: token IDs {input_contract} -> next-token logits {output_contract}")
print("  -> All four internals can change while the causal prediction contract stays fixed.")

## 1. RMSNorm: Preserve Scale Without Recentering

LayerNorm recenters and rescales each token. The modern block asks a narrower question: can Riverside stabilize overall token magnitude without forcing the feature mean to zero?

**Predict:** On the same six token vectors, will RMSNorm force the mean to zero, force root-mean-square magnitude to one, or do both?

```mermaid
flowchart LR
    A["Token vector uneven scale"] --> B["LayerNorm recenter + rescale"] --> D["mean near 0; RMS near 1"]
    A --> C["RMSNorm rescale only"] --> E["offset remains; RMS near 1"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Measure LayerNorm and RMSNorm on identical Riverside activations ---------
torch.manual_seed(SEED)
activations = torch.randn(len(TOKENS), 12) * torch.linspace(0.4, 2.2, len(TOKENS)).unsqueeze(1) + 0.7
layer_normed = F.layer_norm(activations, (12,))
rms_normed = activations / activations.pow(2).mean(-1, keepdim=True).add(1e-5).sqrt()
ln_mean = layer_normed.mean(-1)
rn_mean = rms_normed.mean(-1)
rn_rms = rms_normed.pow(2).mean(-1).sqrt()
norm_stats = {"LayerNorm mean": ln_mean.abs().mean().item(), "RMSNorm RMS": rn_rms.mean().item()}
fig, axes = plt.subplots(2, 2, figsize=(12, 7), facecolor=DARK)
for axis, data, title in zip(axes.flat[:3], [activations, layer_normed, rms_normed], ["Raw", "LayerNorm", "RMSNorm"]):
    image = axis.imshow(data.numpy(), aspect="auto", cmap="coolwarm", vmin=-3, vmax=3)
    axis.set_title(title, color="white")
    axis.set_yticks(range(len(TOKENS)), TOKENS, color="white")
    axis.tick_params(axis="x", colors="white")
    fig.colorbar(image, ax=axis, fraction=0.046)
x = np.arange(len(TOKENS))
axes[1, 1].plot(x, ln_mean.numpy(), marker="o", color=BLUE, label="LayerNorm mean")
axes[1, 1].plot(x, rn_mean.numpy(), marker="o", color=AMBER, label="RMSNorm mean")
axes[1, 1].plot(x, rn_rms.numpy(), marker="s", color=GREEN, label="RMSNorm RMS")
axes[1, 1].set_xticks(x, TOKENS, rotation=25, ha="right", color="white")
axes[1, 1].tick_params(axis="y", colors="white")
axes[1, 1].legend(facecolor=DARK, labelcolor="white")
axes[1, 1].set_title("Per-token contracts", color="white")
fig.tight_layout(); plt.show()
print(f"Mean absolute LayerNorm output mean: {ln_mean.abs().mean():.6f}")
print(f"Mean RMSNorm output RMS: {rn_rms.mean():.6f}")
print("  -> RMSNorm enforces scale without enforcing zero mean.")

**Code Walkthrough: LayerNorm vs. RMSNorm panels**

1. One seeded activation matrix goes through both paths, so only the normalization rule changes.
2. Token identity stays row-aligned across heatmaps.
3. The final panel checks each contract directly instead of inferring behavior from color.
4. This proves behavior, not a universal speed or quality ranking.

In [ ]:
# -- Animate one token vector before and after normalization ------------------
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), facecolor=DARK, sharey=True)
series = [activations, layer_normed, rms_normed]
lines = []
for axis, title, color in zip(axes, ["Before", "LayerNorm", "RMSNorm"], [RED, BLUE, AMBER]):
    line, = axis.plot(range(12), np.zeros(12), color=color, marker="o")
    axis.axhline(0, color="white", linewidth=0.7)
    axis.set_ylim(-4, 4)
    axis.set_title(title, color="white")
    axis.tick_params(colors="white")
    lines.append(line)
title_text = fig.suptitle("", color="white", fontsize=13)
def update_normalization(frame):
    for line, values in zip(lines, series):
        line.set_ydata(values[frame].numpy())
    title_text.set_text(f"Token {frame + 1}: {TOKENS[frame]}")
    return [*lines, title_text]
normalization_animation = FuncAnimation(fig, update_normalization, frames=len(TOKENS), interval=650, blit=False)
normalization_html = normalization_animation.to_jshtml()
plt.close(fig)
print("What to watch: LayerNorm recenters each frame; RMSNorm rescales while retaining the token's offset.")
display(HTML(normalization_html))

In [ ]:
# -- Implement RMSNorm and measure a deep residual gradient -------------------
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        scale = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return x * scale * self.weight

def deep_stack_gradient(norm_factory, depth=24, d_model=32):
    torch.manual_seed(SEED)
    norms = nn.ModuleList([norm_factory(d_model) for _ in range(depth)])
    layers = nn.ModuleList([nn.Linear(d_model, d_model, bias=False) for _ in range(depth)])
    x = torch.randn(2, len(TOKENS), d_model, requires_grad=True)
    hidden = x
    for norm, layer in zip(norms, layers):
        hidden = hidden + 0.1 * torch.tanh(layer(norm(hidden)))
    hidden.square().mean().backward()
    return x.grad.norm().item()

gradient_norms = {
    "LayerNorm": deep_stack_gradient(lambda width: nn.LayerNorm(width)),
    "RMSNorm": deep_stack_gradient(lambda width: RMSNorm(width)),
}
fig, axis = plt.subplots(figsize=(6, 3.6), facecolor=DARK)
axis.bar(gradient_norms.keys(), gradient_norms.values(), color=[BLUE, AMBER])
axis.set_title("24-block seeded residual stack", color="white")
axis.set_ylabel("Input gradient norm", color="white"); axis.tick_params(colors="white")
plt.show()
assert all(value > 0 for value in gradient_norms.values())
print(gradient_norms)
print("  -> Both routes remain differentiable; this supports a simpler contract, not a universal ranking.")

**Code Walkthrough: RMSNorm and the deep-stack check**

1. `RMSNorm.forward` computes one scale per token and leaves recentering out.
2. Every residual update is small so the experiment tests a plausible pre-norm route.
3. Resetting the seed gives both choices identical linear weights and inputs.
4. The assertion asks whether gradients arrive; one toy run does not establish a quality winner.

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Divide by a global batch statistic | Tokens become coupled to unrelated examples |
| Right | Reduce over the final feature axis | Every token owns its scale |
| Wrong | Claim RMSNorm always trains better | The toy check cannot establish model quality |
| Right | State the narrower contract | The conclusion matches the evidence |

**Quick Health Check:** verify shape preservation, unit RMS, finite values, and a trainable scale gradient.

**Your turn:** change only `offset`. Predict which method removes it from the output mean.

```python
# CHANGE THIS: try -2.0 or 8.0
offset = 3.0
```

**Checkpoint:** RMSNorm preserves the residual-stream shape and a usable gradient route without requiring zero-mean features.

In [ ]:
# -- Run RMSNorm health checks and the one-variable exercise ------------------
rms_layer = RMSNorm(12)
health_input = activations.clone().requires_grad_(True)
health_output = rms_layer(health_input)
health_output.sum().backward()
assert health_output.shape == health_input.shape
assert torch.isfinite(health_output).all()
assert torch.allclose(health_output.pow(2).mean(-1).sqrt(), torch.ones(len(TOKENS)), atol=2e-4)
assert rms_layer.weight.grad is not None and rms_layer.weight.grad.abs().sum() > 0
offset = 3.0  # CHANGE THIS: try -2.0 or 8.0
trial = torch.tensor([[1.0, 2.0, 4.0, 7.0]]) + offset
trial_ln = F.layer_norm(trial, (4,))
trial_rn = RMSNorm(4)(trial)
assert abs(trial_ln.mean().item()) < 1e-5
print(f"LayerNorm mean={trial_ln.mean().item():.6f}; RMSNorm mean={trial_rn.mean().item():.6f}")
print("PASS: RMSNorm health checks passed; only LayerNorm guarantees removal of the offset.")

**Reflection:** RMSNorm removed one operation from the normalization contract. It did not make the per-token transformation selective. Riverside still sends every expanded FFN feature through one activation path.

## 2. SwiGLU: Make the Feed-Forward Network Choose What Passes

The existing GELU FFN expands a token, activates one branch, and contracts it. SwiGLU splits expansion into candidate content and an independent learned gate, multiplies them feature by feature, then projects back to model width.

**Predict:** If candidate content stays fixed while the gate changes, can an expanded feature shrink, grow, or reverse sign?

```mermaid
flowchart LR
    A["Token vector"] --> B["Candidate projection"] --> D["Feature-wise gate"]
    A --> C["Gate projection + SiLU"] --> D
    D --> E["Output projection"] --> F["Residual stream"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Animate candidate content, learned gate, and gated output ----------------
torch.manual_seed(SEED)
gate_input = torch.randn(len(TOKENS), 8)
W_candidate = torch.randn(8, 10) / math.sqrt(8)
W_gate = torch.randn(8, 10) / math.sqrt(8)
candidate_values = gate_input @ W_candidate
gate_values = F.silu(gate_input @ W_gate)
gated_values = candidate_values * gate_values
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), facecolor=DARK, sharey=True)
lines = []
for axis, title, color in zip(axes, ["Candidate", "Learned gate", "Gated result"], [BLUE, AMBER, GREEN]):
    line, = axis.plot(range(10), np.zeros(10), marker="o", color=color)
    axis.axhline(0, color="white", linewidth=0.7); axis.set_ylim(-4, 4)
    axis.set_title(title, color="white"); axis.tick_params(colors="white")
    lines.append(line)
title_text = fig.suptitle("", color="white", fontsize=13)
def update_gate(frame):
    for line, values in zip(lines, [candidate_values, gate_values, gated_values]):
        line.set_ydata(values[frame].numpy())
    title_text.set_text(f"Token {frame + 1}: {TOKENS[frame]}")
    return [*lines, title_text]
gate_animation = FuncAnimation(fig, update_gate, frames=len(TOKENS), interval=650, blit=False)
gate_html = gate_animation.to_jshtml()
plt.close(fig)
print("What to watch: the candidate stays identifiable while the gate suppresses, preserves, or reverses features.")
display(HTML(gate_html))

In [ ]:
# -- Ablate the gate with fixed weights, then implement SwiGLU ----------------
torch.manual_seed(SEED)
ablation_input = torch.randn(4, 8)
W_down = torch.randn(10, 8) / math.sqrt(10)
candidate = ablation_input @ W_candidate
gate = F.silu(ablation_input @ W_gate)
swiglu_output = (candidate * gate) @ W_down
gate_ablated_output = candidate @ W_down
gelu_baseline = F.gelu(candidate) @ W_down
gate_delta = (swiglu_output - gate_ablated_output).norm().item()
print(f"Gate ablation delta={gate_delta:.4f}; GELU baseline delta={(swiglu_output - gelu_baseline).norm():.4f}")
assert gate_delta > 0

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        gate = F.silu(self.gate_proj(x))
        candidate = self.up_proj(x)
        return self.down_proj(gate * candidate)

swiglu = SwiGLU(8, 16)
assert swiglu(ablation_input).shape == ablation_input.shape
print("  -> The independent gate is active, and the module returns to model width.")

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Reuse one projection for gate and candidate | Removes the independent decision |
| Right | Separate `gate_proj` and `up_proj` | Content and passage control can differ |
| Wrong | Add the branches | No feature-wise gate remains |
| Right | Multiply before `down_proj` | Selection happens before contraction |

**Quick Health Check:** verify width, separate storage, a nonzero ablation delta, and gradients on all three projections.

**Your turn:** set `fixed_gate` to `0.0`, `1.0`, or `-0.5`. Predict the output norm and sign behavior before running.

**Reflection:** SwiGLU made the per-token transform selective. It did nothing about attention memory: eight query heads still imply eight key/value stores in MHA.

In [ ]:
# -- Run SwiGLU health checks and the controlled-gate exercise ----------------
health_swiglu = SwiGLU(8, 16)
health_gate_input = torch.randn(2, 3, 8, requires_grad=True)
health_gate_output = health_swiglu(health_gate_input)
health_gate_output.square().mean().backward()
assert health_gate_output.shape == health_gate_input.shape
assert health_swiglu.gate_proj.weight.data_ptr() != health_swiglu.up_proj.weight.data_ptr()
for projection in [health_swiglu.gate_proj, health_swiglu.up_proj, health_swiglu.down_proj]:
    assert projection.weight.grad is not None and projection.weight.grad.abs().sum() > 0
fixed_gate = 0.25  # CHANGE THIS: try 0.0, 1.0, or -0.5
controlled = (candidate * fixed_gate) @ W_down
expected = abs(fixed_gate) * (candidate @ W_down).norm()
assert torch.allclose(controlled.norm(), expected, atol=1e-6)
print(f"Controlled output norm={controlled.norm():.6f}")
print("PASS: SwiGLU preserves width, uses separate branches, and all projections receive gradients.")

## 3. Grouped-Query Attention: Many Questions, Fewer Stored Answers

Eight query heads let Riverside ask eight different questions about `Aria heard the signal aboard Meridian`. MHA gives each question its own key and value projections. MQA shares one pair across all questions. GQA shares within groups.

**Predict:** With 8 query heads and 4 key/value heads, how many queries should each group serve?

```mermaid
flowchart TB
    Q0["Q0 Q1"] --> K0["KV group 0"]
    Q1["Q2 Q3"] --> K1["KV group 1"]
    Q2["Q4 Q5"] --> K2["KV group 2"]
    Q3["Q6 Q7"] --> K3["KV group 3"]
    style Q0 fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Q1 fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Q2 fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Q3 fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style K0 fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style K1 fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style K2 fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style K3 fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Draw the GQA ownership matrix -------------------------------------------
n_query_heads, n_kv_heads = 8, 4
queries_per_group = n_query_heads // n_kv_heads
ownership = np.zeros((n_query_heads, n_kv_heads), dtype=int)
for query_head in range(n_query_heads):
    ownership[query_head, query_head // queries_per_group] = 1
fig, axis = plt.subplots(figsize=(7, 5), facecolor=DARK)
axis.imshow(ownership, cmap="Blues", vmin=0, vmax=1, aspect="auto")
axis.set_xticks(range(n_kv_heads), [f"KV {i}" for i in range(n_kv_heads)], color="white")
axis.set_yticks(range(n_query_heads), [f"Q {i}" for i in range(n_query_heads)], color="white")
axis.set_xlabel("Shared key/value group", color="white"); axis.set_ylabel("Query head", color="white")
axis.set_title("GQA ownership: 8 queries, 4 KV groups", color="white")
for row in range(n_query_heads):
    for column in range(n_kv_heads):
        axis.text(column, row, str(ownership[row, column]), ha="center", va="center", color="white")
plt.show()
assert ownership.sum(1).tolist() == [1] * n_query_heads
assert ownership.sum(0).tolist() == [2] * n_kv_heads
print("  -> Each query owns one group; each key/value group serves two queries.")

In [ ]:
# -- Animate query heads fanning into shared key/value groups -----------------
fig, axis = plt.subplots(figsize=(10, 5), facecolor=DARK)
query_x = np.linspace(0.08, 0.92, n_query_heads)
kv_x = np.linspace(0.16, 0.84, n_kv_heads)
axis.scatter(query_x, np.full(n_query_heads, 0.78), s=650, color=BLUE, edgecolor="white")
axis.scatter(kv_x, np.full(n_kv_heads, 0.22), s=850, color=AMBER, edgecolor="white")
for index, x_pos in enumerate(query_x): axis.text(x_pos, 0.78, f"Q{index}", ha="center", va="center", color="white")
for index, x_pos in enumerate(kv_x): axis.text(x_pos, 0.22, f"KV{index}", ha="center", va="center", color="white")
connections = []
for query_head, x_pos in enumerate(query_x):
    group = query_head // queries_per_group
    line, = axis.plot([x_pos, kv_x[group]], [0.70, 0.30], color="#475569", linewidth=1.5)
    connections.append(line)
status = axis.text(0.5, 0.95, "", ha="center", color="white", fontsize=13)
axis.set_xlim(0, 1); axis.set_ylim(0.05, 1.0); axis.axis("off")
def update_sharing(frame):
    for index, line in enumerate(connections):
        line.set_color(GREEN if index == frame else "#475569")
        line.set_linewidth(4 if index == frame else 1.5)
    status.set_text(f"Q{frame} reads KV group {frame // queries_per_group}")
    return [*connections, status]
sharing_animation = FuncAnimation(fig, update_sharing, frames=n_query_heads, interval=500, blit=False)
sharing_html = sharing_animation.to_jshtml()
plt.close(fig)
print("What to watch: query identity stays separate while pairs converge on the same stored key/value group.")
display(HTML(sharing_html))

In [ ]:
# -- Measure MHA, GQA, and MQA K/V parameters and cache elements -------------
d_model, head_dim = 64, 64 // n_query_heads
def measure_kv(n_kv):
    key_projection = nn.Linear(d_model, n_kv * head_dim, bias=False)
    value_projection = nn.Linear(d_model, n_kv * head_dim, bias=False)
    parameters = sum(p.numel() for layer in [key_projection, value_projection] for p in layer.parameters())
    return parameters, 2 * n_kv * head_dim
variants = {"MHA (8 KV)": 8, "GQA (4 KV)": 4, "MQA (1 KV)": 1}
kv_measurements = {name: measure_kv(count) for name, count in variants.items()}
labels = list(kv_measurements)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), facecolor=DARK)
axes[0].bar(labels, [kv_measurements[x][0] for x in labels], color=[BLUE, AMBER, GREEN])
axes[1].bar(labels, [kv_measurements[x][1] for x in labels], color=[BLUE, AMBER, GREEN])
for axis, title, ylabel in zip(axes, ["Measured K/V projection parameters", "Measured cache elements per token"], ["Parameters", "Elements"]):
    axis.set_title(title, color="white"); axis.set_ylabel(ylabel, color="white")
    axis.tick_params(axis="x", rotation=20, colors="white"); axis.tick_params(axis="y", colors="white")
fig.tight_layout(); plt.show()
for label in labels:
    print(f"{label:12s}: K/V params={kv_measurements[label][0]:5d}; cache elements/token={kv_measurements[label][1]:3d}")
gqa_cache_ratio = kv_measurements["GQA (4 KV)"][1] / kv_measurements["MHA (8 KV)"][1]
print(f"  -> This GQA layout stores {gqa_cache_ratio:.0%} of MHA K/V cache elements per token.")

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Leave a partial query head | Head reshaping is invalid |
| Right | Require model width divisible by query heads | Every query gets equal width |
| Wrong | Split queries unevenly across KV groups | Sharing cannot map consistently |
| Right | Require query heads divisible by KV heads | Every group serves an integer count |

**Quick Health Check:** run both divisibility assertions and prove an invalid grouping fails explicitly.

**Your turn:** keep eight query heads and choose `trial_kv_heads` from 1, 2, 4, or 8. Predict cache elements before running.

In [ ]:
# -- Validate GQA dimensions and expose the invalid case ----------------------
def validate_gqa_dimensions(model_width, query_heads, kv_heads):
    assert model_width % query_heads == 0, "d_model must be divisible by n_query_heads"
    assert query_heads % kv_heads == 0, "n_query_heads must be divisible by n_kv_heads"
validate_gqa_dimensions(64, 8, 4)
try:
    validate_gqa_dimensions(64, 8, 3)
except AssertionError as error:
    print(f"EXPECTED FAILURE: invalid GQA grouping rejected - {error}")
else:
    raise AssertionError("Invalid GQA grouping was not rejected")
trial_kv_heads = 2  # CHANGE THIS: valid choices are 1, 2, 4, 8
validate_gqa_dimensions(d_model, n_query_heads, trial_kv_heads)
trial_parameters, trial_cache = measure_kv(trial_kv_heads)
assert trial_cache == 2 * trial_kv_heads * head_dim
print(f"{trial_kv_heads} KV heads -> {trial_parameters} K/V parameters; {trial_cache} cache elements/token")
print("PASS: valid dimensions proceed and invalid grouping fails explicitly.")

**Reflection:** GQA preserves eight independent query projections while halving this configured model's K/V projection parameters and per-token cache elements relative to MHA. The ownership map says who shares; it does not say where each token sits.

## 4. RoPE in the Real Attention Path

Part 1 already built RoPE intuition. Here the architectural change is placement: rotate queries and keys after head splitting and before similarity scores. Values are not rotated.

**Predict:** If both query and key positions shift equally while their gap stays fixed, should their selected dot product change?

```mermaid
flowchart LR
    X["Normalized token states"] --> Q["Query heads"] --> RQ["RoPE"] --> S["Causal scores"]
    X --> K["Shared key heads"] --> RK["RoPE"] --> S
    X --> V["Shared value heads"] --> O["Weighted values"]
    S --> O
    style X fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Q fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style K fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style V fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style RQ fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style RK fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Implement RoPE helpers and test their critical invariants ----------------
def build_rope_cache(sequence_length, head_width, base=10000.0, device=None):
    assert head_width % 2 == 0, "RoPE requires an even head width"
    pair_indices = torch.arange(0, head_width, 2, device=device, dtype=torch.float32)
    frequencies = 1.0 / (base ** (pair_indices / head_width))
    angles = torch.outer(torch.arange(sequence_length, device=device, dtype=torch.float32), frequencies)
    return angles.cos(), angles.sin()

def apply_rope(x, cos, sin):
    even, odd = x[..., 0::2], x[..., 1::2]
    cos = cos.view(1, 1, cos.shape[0], cos.shape[1])
    sin = sin.view(1, 1, sin.shape[0], sin.shape[1])
    rotated_even = even * cos - odd * sin
    rotated_odd = even * sin + odd * cos
    return torch.stack((rotated_even, rotated_odd), dim=-1).flatten(-2)

torch.manual_seed(SEED)
rope_sample = torch.randn(2, 4, len(TOKENS), 8)
rope_cos, rope_sin = build_rope_cache(16, 8)
rotated = apply_rope(rope_sample, rope_cos[:len(TOKENS)], rope_sin[:len(TOKENS)])
rope_norm_error = (rope_sample.norm(dim=-1) - rotated.norm(dim=-1)).abs().max().item()
query, key = torch.randn(1, 1, 1, 8), torch.randn(1, 1, 1, 8)
dot_2_7 = (apply_rope(query, rope_cos[2:3], rope_sin[2:3]) * apply_rope(key, rope_cos[7:8], rope_sin[7:8])).sum()
dot_5_10 = (apply_rope(query, rope_cos[5:6], rope_sin[5:6]) * apply_rope(key, rope_cos[10:11], rope_sin[10:11])).sum()
rope_shift_error = (dot_2_7 - dot_5_10).abs().item()
assert rope_norm_error < 1e-5 and rope_shift_error < 1e-5
print(f"Maximum norm change={rope_norm_error:.8f}; equal-shift dot change={rope_shift_error:.8f}")
print("PASS: RoPE preserves vector norms and the selected relative-gap dot product.")

**Code Walkthrough: RoPE helpers**

1. The cache holds reusable rotation values for each position and adjacent feature pair.
2. `apply_rope` rotates adjacent pairs and restores the original head shape.
3. One test proves direction changes without length changes.
4. Positions 2/7 and 5/10 have equal gaps; their selected query-key dot products must match.

**Warning:** Apply RoPE after splitting query and key heads. Rotating values changes retrieved content; rotating before head splitting can pair features across the wrong boundary.

**Checkpoint:** Position now lives inside attention without a learned position table.

## 5. Assemble `ModernDecoderBlock` and `TinyModernLM`

Assembly is intentionally plain: RMSNorm before each sublayer, a residual after GQA with RoPE, another RMSNorm, then a residual after SwiGLU.

```mermaid
flowchart LR
    A["IDs (B,T)"] --> B["Embedding (B,T,64)"] --> C["2 x ModernDecoderBlock"] --> D["Final RMSNorm"] --> E["Tied logits (B,T,64)"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Define the model contract and grouped-query causal attention -------------
@dataclass
class TinyModernConfig:
    architecture: str = "tiny_modern_decoder"
    vocab_size: int = 64
    context_length: int = 32
    d_model: int = 64
    n_layers: int = 2
    n_query_heads: int = 8
    n_kv_heads: int = 4
    d_ff: int = 176
    rope_base: float = 10000.0
    rms_norm_eps: float = 1e-5
    tie_embeddings: bool = True

class GroupedQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        validate_gqa_dimensions(config.d_model, config.n_query_heads, config.n_kv_heads)
        self.n_query_heads, self.n_kv_heads = config.n_query_heads, config.n_kv_heads
        self.head_dim = config.d_model // config.n_query_heads
        self.queries_per_group = config.n_query_heads // config.n_kv_heads
        self.rope_base = config.rope_base
        self.q_proj = nn.Linear(config.d_model, config.n_query_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.d_model, config.d_model, bias=False)
        self.last_attention = None

    def _split_heads(self, x, n_heads):
        batch, tokens, _ = x.shape
        return x.view(batch, tokens, n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x):
        batch, tokens, _ = x.shape
        query = self._split_heads(self.q_proj(x), self.n_query_heads)
        key = self._split_heads(self.k_proj(x), self.n_kv_heads)
        value = self._split_heads(self.v_proj(x), self.n_kv_heads)
        cos, sin = build_rope_cache(tokens, self.head_dim, self.rope_base, x.device)
        query, key = apply_rope(query, cos, sin), apply_rope(key, cos, sin)
        key = key.repeat_interleave(self.queries_per_group, dim=1)
        value = value.repeat_interleave(self.queries_per_group, dim=1)
        scores = query @ key.transpose(-2, -1) / math.sqrt(self.head_dim)
        mask = torch.triu(torch.ones(tokens, tokens, dtype=torch.bool, device=x.device), diagonal=1)
        attention = scores.masked_fill(mask, torch.finfo(scores.dtype).min).softmax(dim=-1)
        self.last_attention = attention.detach()
        mixed = (attention @ value).transpose(1, 2).contiguous().view(batch, tokens, -1)
        return self.o_proj(mixed)

**Code Walkthrough: GroupedQueryAttention**

1. Query projection width stays at model width; key and value widths shrink with `n_kv_heads`.
2. RoPE is applied after splitting heads and only to queries and keys.
3. `repeat_interleave` gives each query its group's K/V tensors without adding learned projections.
4. The strict upper triangle blocks future keys before softmax.
5. Heads return to `(batch, tokens, d_model)` before the output projection.

In [ ]:
# -- Assemble the modern block and compact decoder-only language model --------
class ModernDecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn_norm = RMSNorm(config.d_model, config.rms_norm_eps)
        self.attn = GroupedQueryAttention(config)
        self.ffn_norm = RMSNorm(config.d_model, config.rms_norm_eps)
        self.ffn = SwiGLU(config.d_model, config.d_ff)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        return x + self.ffn(self.ffn_norm(x))

class TinyModernLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.blocks = nn.ModuleList([ModernDecoderBlock(config) for _ in range(config.n_layers)])
        self.final_norm = RMSNorm(config.d_model, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        if config.tie_embeddings:
            self.lm_head.weight = self.token_embedding.weight

    def forward(self, token_ids):
        if token_ids.shape[1] > self.config.context_length:
            raise ValueError("Sequence exceeds configured context_length")
        hidden = self.token_embedding(token_ids)
        for block in self.blocks:
            hidden = block(hidden)
        return self.lm_head(self.final_norm(hidden))

torch.manual_seed(SEED)
config = TinyModernConfig()
model = TinyModernLM(config)
print(model)
print("  -> The copyable reference contains the complete modern dense decoder path.")

**Code Walkthrough: ModernDecoderBlock and TinyModernLM**

1. Both sublayers use pre-norm and return to the unchanged residual stream.
2. `ModuleList` registers blocks while keeping the forward loop readable.
3. No learned position embedding appears because RoPE operates inside attention.
4. Assigning `lm_head.weight` shares one parameter object; the next diagnostics inspect storage, not just values.
5. The context check fails before reshape or mask errors obscure the real problem.

In [ ]:
# -- Generate actual-dimension tensor flow from an executed forward pass ------
token_ids = torch.tensor([[3, 9, 12, 5, 13, 7], [3, 9, 12, 5, 13, 7]])
shape_trace = {}
hooks = [
    model.token_embedding.register_forward_hook(lambda m, i, o: shape_trace.update(embedding=tuple(o.shape))),
    model.blocks[0].attn.register_forward_hook(lambda m, i, o: shape_trace.update(gqa=tuple(o.shape))),
    model.blocks[0].ffn.register_forward_hook(lambda m, i, o: shape_trace.update(swiglu=tuple(o.shape))),
    model.final_norm.register_forward_hook(lambda m, i, o: shape_trace.update(final_norm=tuple(o.shape))),
]
logits = model(token_ids)
for hook in hooks: hook.remove()
shape_trace.update(ids=tuple(token_ids.shape), logits=tuple(logits.shape))
flow = [("Token IDs", shape_trace["ids"]), ("Embedding", shape_trace["embedding"]), ("GQA", shape_trace["gqa"]), ("SwiGLU", shape_trace["swiglu"]), ("Final norm", shape_trace["final_norm"]), ("Logits", shape_trace["logits"])]
fig, axis = plt.subplots(figsize=(14, 3.2), facecolor=DARK)
for index, (label, shape) in enumerate(flow):
    color = GREEN if label == "Logits" else BLUE if index else AMBER
    axis.text(index, 0.5, f"{label}\n{shape}", ha="center", va="center", color="white", bbox={"boxstyle": "round,pad=0.5", "facecolor": color, "edgecolor": "white"})
    if index < len(flow) - 1:
        axis.annotate("", xy=(index + 0.72, 0.5), xytext=(index + 0.28, 0.5), arrowprops={"arrowstyle": "->", "color": "white", "lw": 2})
axis.set_xlim(-0.6, len(flow) - 0.4); axis.set_ylim(0, 1); axis.axis("off")
axis.set_title("TinyModernLM tensor flow from the executed pass", color="white")
plt.show()
print(f"Forward contract: {shape_trace['ids']} -> {shape_trace['logits']}")
print("  -> Every residual branch returns to model width; only the vocabulary head changes the final axis.")

In [ ]:
# -- Visualize component parameters from named_parameters ---------------------
component_counts = {"embedding/tied head": 0, "attention": 0, "SwiGLU": 0, "norms": 0}
for name, parameter in model.named_parameters():
    if name.startswith("token_embedding"): component_counts["embedding/tied head"] += parameter.numel()
    elif ".attn." in name: component_counts["attention"] += parameter.numel()
    elif ".ffn." in name: component_counts["SwiGLU"] += parameter.numel()
    elif "norm" in name: component_counts["norms"] += parameter.numel()
fig, axis = plt.subplots(figsize=(10, 2.8), facecolor=DARK)
left = 0
for (label, count), color in zip(component_counts.items(), [BLUE, AMBER, GREEN, RED]):
    axis.barh([0], [count], left=left, label=f"{label}: {count:,}", color=color)
    left += count
axis.set_xlabel("Unique trainable parameters", color="white"); axis.set_yticks([]); axis.tick_params(colors="white")
axis.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, facecolor=DARK, labelcolor="white")
axis.set_title("TinyModernLM component accounting", color="white")
plt.show()
unique_parameter_total = sum(parameter.numel() for parameter in model.parameters())
assert sum(component_counts.values()) == unique_parameter_total
print(f"Unique trainable parameters: {unique_parameter_total:,}")
print("  -> The tied vocabulary head adds no second parameter matrix.")

### Required Diagnostics

A clean forward shape can hide a broken model. The next cell tests storage identity, every forbidden future-mask entry, and gradient reachability across normalization, attention, and SwiGLU.

**Predict:** Equal embedding and output values are not enough to prove weight tying. Which property must match: shape, values, Python storage address, or all three?

In [ ]:
# -- Run tied-storage, causal-mask, and gradient diagnostics ------------------
tied_storage = model.token_embedding.weight.data_ptr() == model.lm_head.weight.data_ptr()
assert tied_storage
future_positions = torch.triu(torch.ones(len(TOKENS), len(TOKENS), dtype=torch.bool), diagonal=1)
future_attention = model.blocks[0].attn.last_attention[..., future_positions]
causal_leak = future_attention.abs().max().item()
assert causal_leak == 0.0
targets, next_token_logits = token_ids[:, 1:], logits[:, :-1, :]
loss = F.cross_entropy(next_token_logits.reshape(-1, config.vocab_size), targets.reshape(-1))
model.zero_grad(set_to_none=True); loss.backward()
gradient_paths = {
    "RMSNorm": model.blocks[0].attn_norm.weight,
    "query projection": model.blocks[0].attn.q_proj.weight,
    "key projection": model.blocks[0].attn.k_proj.weight,
    "value projection": model.blocks[0].attn.v_proj.weight,
    "output projection": model.blocks[0].attn.o_proj.weight,
    "SwiGLU gate": model.blocks[0].ffn.gate_proj.weight,
    "SwiGLU candidate": model.blocks[0].ffn.up_proj.weight,
    "SwiGLU down": model.blocks[0].ffn.down_proj.weight,
}
for label, parameter in gradient_paths.items():
    gradient_norm = 0.0 if parameter.grad is None else parameter.grad.norm().item()
    assert gradient_norm > 0, f"No gradient reached {label}"
    print(f"{label:20s} gradient norm: {gradient_norm:.6f}")
print(f"Tied storage={tied_storage}; maximum future attention={causal_leak:.1f}; loss={loss.item():.4f}")
print("PASS: storage is tied, all future keys are blocked, and gradients reach every required path.")

**Code Walkthrough: model diagnostics**

1. `data_ptr()` checks shared storage; equal initial values would be too weak.
2. The strict upper triangle selects every forbidden attention entry, whose maximum must be zero.
3. Shifted Riverside token IDs create a real next-token loss.
4. One backward pass reports each required parameter route so a future refactor reveals exactly what broke.

**Checkpoint:** The model now passes shape, tied-storage, causal-mask, RoPE, divisibility, and gradient checks.

## 6. Toy-to-Real Configuration Bridge

The table is a **representative open Llama-style mapping**, not a downloaded model config and not a claim about one checkpoint. Field names vary across families; these common names are enough to connect the teaching model to production documentation without network access.

```mermaid
flowchart LR
    A["Teaching config"] --> B["Validate dimensions"] --> C["Map Llama-style fields"] --> D["Write model-config.json"] --> E["Notebook 06 reconstructs model"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Teaching field | Representative Llama-style field | Meaning |
|---|---|---|
| `vocab_size` | `vocab_size` | embedding and output rows |
| `context_length` | `max_position_embeddings` | maximum token window |
| `d_model` | `hidden_size` | residual-stream width |
| `n_layers` | `num_hidden_layers` | decoder block count |
| `n_query_heads` | `num_attention_heads` | independent query heads |
| `n_kv_heads` | `num_key_value_heads` | shared key/value groups |
| `d_ff` | `intermediate_size` | SwiGLU expanded width |
| `rope_base` | `rope_theta` | RoPE frequency base |
| `rms_norm_eps` | `rms_norm_eps` | normalization stability constant |
| `tie_embeddings` | `tie_word_embeddings` | shared embedding/head storage |

In [ ]:
# -- Validate and write the frozen model configuration contract --------------
model_config = asdict(config)
required_config_keys = {
    "architecture", "vocab_size", "context_length", "d_model", "n_layers",
    "n_query_heads", "n_kv_heads", "d_ff", "rope_base", "rms_norm_eps", "tie_embeddings",
}
assert set(model_config) == required_config_keys
validate_gqa_dimensions(model_config["d_model"], model_config["n_query_heads"], model_config["n_kv_heads"])
artifact_directory = Path("artifacts/base-lm")
artifact_directory.mkdir(parents=True, exist_ok=True)
config_path = artifact_directory / "model-config.json"
config_path.write_text(json.dumps(model_config, indent=2) + "\n", encoding="utf-8")
reloaded_config = json.loads(config_path.read_text(encoding="utf-8"))
assert reloaded_config == model_config
print(f"Wrote and reloaded: {config_path.resolve()}")
print("PASS: the artifact contains every frozen cross-notebook model field.")

## 7. Closing Scorecard

The opening blocker was architectural, not objective-level. The same causal next-token task now runs through a decoder whose internal choices match the vocabulary of a modern dense Llama-style model.

| Opening question | Measured answer from this notebook |
|---|---|
| Can normalization avoid recentering while controlling token scale? | RMSNorm held each token's root-mean-square scale at the configured target, and gradients remained finite through the comparison stack. |
| Can the FFN learn what content passes forward? | SwiGLU exposed separate candidate and gate branches; the measured ablation changed the output when the gate was removed. |
| Can query heads share stored keys and values? | MHA, GQA, and MQA used the same query-head count while measured K/V parameters and cache elements fell as sharing increased. |
| Can position live inside attention? | RoPE preserved vector norms and preserved the selected Q/K score when both positions shifted by the same amount. |
| Does the assembled model preserve causal training? | Future attention remained exactly blocked, every required parameter route received gradient, and input/output embedding storage stayed tied. |
| Can the next notebook rebuild the same architecture? | `model-config.json` was written and reloaded with every frozen cross-notebook field. |

**Key takeaways**

- Modern decoder families keep the same next-token objective while changing how blocks normalize, gate features, share K/V state, and represent position.
- GQA is a sharing decision: query heads remain distinct while groups reuse key and value projections.
- SwiGLU makes the FFN's gate visible; it does not turn the FFN into attention or an expert router.
- RoPE belongs on projected queries and keys, where attention compares positions.
- A configuration artifact is part of the model contract: Notebook 06 should not guess architecture dimensions.

### Coverage ledger

| Tier | Covered here | Boundary |
|---|---|---|
| Implemented and measured | RMSNorm, SwiGLU, GQA/MHA/MQA comparison, RoPE in attention, causal masking, tied embeddings, `ModernDecoderBlock`, `TinyModernLM`, parameter accounting, config export | Every claim is backed by this notebook's tensors and diagnostics. |
| Explained but not production-implemented | Representative Llama-style config mapping | The notebook maps field meanings without downloading or claiming parity with one remote checkpoint. |
| Named and out of scope | FlashAttention, sliding-window attention, mixture-of-experts, quantization, distributed training, production kernels | These change efficiency or capacity at scale; they are not required to understand how a dense base decoder is built. |

**Checkpoint:** Notebook 06 may replace `vocab_size` with the tokenizer's actual vocabulary size, but it must preserve the remaining architecture contract.